### Algorithm from Scratch

In [1]:
import torch
import torch.nn as nn


In [56]:
# class for the backbone
from torchvision import models
class Backbone(nn.Module):
    def __init__(self, d_model: int = 256):
        super().__init__()
        resnet = models.resnet50(weights = models.ResNet50_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])
        # print(self.backbone.children)
        self.projection = nn.Conv2d(2048,d_model,1)


    def forward(self,x):
        x = self.backbone(x)
        x = self.projection(x)
        # print(x.shape)
        return x

test = torch.randn((2,3,800,800))
test = Backbone()(test)
print(test.shape)

torch.Size([2, 256, 25, 25])


In [ ]:
import math
import matplotlib.pyplot as plt

class PositionalEncoding2D(nn.Module):
    def __init__(self,d_model: int = 256,
                 temperature: int = 10000):
        super().__init__()
        assert d_model % 2 == 0, "Provide the Even dimension as positional Dimension"
        self.d_model = d_model
        self.d_half = d_model // 2
        self.temperature = temperature


        i = torch.arange(self.d_half//2 , dtype=torch.float32)
        freq = self.temperature**(2*i / self.d_half)

        self.register_buffer('freq',freq)
        
    def _compute_1d_pe(self, positions: torch.Tensor) -> torch.Tensor:
        angles = positions[:, None] / self.freq[None,:]

        pe = torch.zeros(len(positions), self.d_half,
                         device= positions.device, dtype= torch.float32)
        
        pe[:, 0::2] = torch.sin(angles)
        pe[:, 1::2] = torch.cos(angles)

        return pe


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape

        y_position = torch.arange(h, device=x.device, dtype=torch.float32)
        x_position = torch.arange(w, device=x.device, dtype=torch.float32)

        y_pe = self._compute_1d_pe(y_position)
        x_pe = self._compute_1d_pe(x_position)

        y_broadcast = y_pe[:, None, :].expand(h,w,self.d_half)
        x_broadcast = x_pe[None, :, :].expand(h,w,self.d_half)

        pe = torch.cat([y_broadcast,x_broadcast], dim = -1)
        pe = pe.permute(2,0,1).unsqueeze(0)

        return x + pe


In [70]:
pe_module = PositionalEncoding2D(d_model=256)
dummy     = torch.zeros(1, 256, 15, 15)
output    = pe_module(dummy)    # (1, 256, 15, 15)

# Extract the PE (since input was zeros, output IS the PE)
pe = output[0]    # (256, 15, 15)

# Test 1: two different rows at same column should be different
assert not torch.allclose(pe[:, 0, 0], pe[:, 1, 0]), \
    "Row 0 and Row 1 should have different encodings"

# Test 2: two different cols at same row should be different  
assert not torch.allclose(pe[:, 0, 0], pe[:, 0, 1]), \
    "Col 0 and Col 1 should have different encodings"

# Test 3: same position across batch should be identical
dummy2   = torch.zeros(4, 256, 15, 15)
output2  = pe_module(dummy2)
assert torch.allclose(output2[0], output2[1]), \
    "Same position should get same PE regardless of batch index"

print("All assertions passed — PE is working correctly")

All assertions passed — PE is working correctly
